# new-thyroid1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('24_new-thyroid1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.269,94.051,97.50,91.429,99.444,95.437,99.643,99.921,94.660
smote,99.580,98.000,96.25,100.000,99.167,99.583,99.260,99.841,98.790
bsmote,99.441,97.333,95.00,100.000,98.889,99.444,99.235,99.841,98.387
ros,99.580,98.000,96.25,100.000,99.167,99.583,99.439,99.881,98.790
adasyn,98.699,96.564,95.00,98.571,98.889,98.730,99.235,99.841,97.631
casb,99.580,98.000,96.25,100.000,99.167,99.583,99.643,99.921,98.790
casb_relaxed,99.441,97.333,95.00,100.000,98.889,99.444,99.235,99.841,98.387
casb_edited,99.441,97.333,95.00,100.000,98.889,99.444,99.439,99.881,98.387
casb_ros,98.839,97.231,96.25,98.571,99.167,98.869,99.439,99.881,98.035


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,90.580,88.938,97.143,82.857,99.444,91.151,99.643,99.921,89.759
smote,98.429,95.238,92.321,98.571,98.333,98.452,96.990,99.623,96.834
bsmote,98.133,94.133,90.750,98.571,97.778,98.175,95.918,99.524,96.133
ros,97.688,94.469,92.321,97.143,98.333,97.738,96.786,99.583,96.078
adasyn,98.429,95.238,92.321,98.571,98.333,98.452,97.143,99.643,96.834
casb,99.161,96.000,92.500,100.000,98.333,99.167,100.000,100.000,97.580
casb_relaxed,99.161,96.000,92.500,100.000,98.333,99.167,99.082,99.802,97.580
casb_edited,98.419,95.231,92.500,98.571,98.333,98.452,99.464,99.881,96.825
casb_ros,97.531,94.030,92.000,97.143,98.056,97.599,98.750,99.861,95.781


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.549,95.385,100.000,91.429,100.000,95.714,99.821,99.960,95.467
smote,96.097,95.366,98.571,92.857,99.722,96.290,99.617,99.921,95.731
bsmote,94.408,93.033,97.321,90.000,99.444,94.722,99.325,99.841,93.721
ros,94.965,92.707,94.643,91.429,98.889,95.159,98.968,99.762,93.836
adasyn,95.946,94.692,97.500,92.857,99.444,96.151,99.439,99.881,95.319
casb,94.538,93.692,98.750,90.000,99.722,94.861,99.504,99.881,94.115
casb_relaxed,95.215,93.930,97.321,91.429,99.444,95.437,99.643,99.921,94.573
casb_edited,95.215,93.930,97.321,91.429,99.444,95.437,99.260,99.841,94.573
casb_ros,95.159,93.495,96.071,91.429,99.167,95.298,99.504,99.881,94.327


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.549,95.385,100.00,91.429,100.000,95.714,100.0,100.0,95.467
smote,98.839,97.231,96.25,98.571,99.167,98.869,100.0,100.0,98.035
bsmote,98.699,96.564,95.00,98.571,98.889,98.730,100.0,100.0,97.631
ros,98.097,96.462,96.25,97.143,99.167,98.155,100.0,100.0,97.279
adasyn,98.699,96.564,95.00,98.571,98.889,98.730,100.0,100.0,97.631
casb,98.839,97.231,96.25,98.571,99.167,98.869,100.0,100.0,98.035
casb_relaxed,98.839,97.231,96.25,98.571,99.167,98.869,100.0,100.0,98.035
casb_edited,98.839,97.231,96.25,98.571,99.167,98.869,100.0,100.0,98.035
casb_ros,98.839,97.231,96.25,98.571,99.167,98.869,100.0,100.0,98.035


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.237,93.440,98.661,89.286,99.722,94.504,99.777,99.950,93.838
smote,98.236,96.459,95.848,97.500,99.097,98.299,98.967,99.846,97.348
bsmote,97.670,95.266,94.518,96.786,98.750,97.768,98.620,99.802,96.468
ros,97.582,95.409,94.866,96.429,98.889,97.659,98.798,99.807,96.496
adasyn,97.943,95.765,94.955,97.143,98.889,98.016,98.954,99.841,96.854
casb,98.029,96.231,95.938,97.143,99.097,98.120,99.787,99.950,97.130
casb_relaxed,98.164,96.124,95.268,97.500,98.958,98.229,99.490,99.891,97.144
casb_edited,97.978,95.931,95.268,97.143,98.958,98.051,99.541,99.901,96.955
casb_ros,97.592,95.497,95.143,96.429,98.889,97.659,99.423,99.906,96.544


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,99.58,smote,99.58,0.0,TIE
1,f1,casb,98.00,smote,98.00,0.0,TIE
2,precision,casb,96.25,smote,96.25,0.0,TIE
3,recall,casb,100.00,smote,100.00,0.0,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,99.161,smote,98.429,0.731,WIN
1,f1,casb,96.000,smote,95.238,0.762,WIN
2,precision,casb,92.500,smote,92.321,0.179,WIN
3,recall,casb,100.000,smote,98.571,1.429,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,95.215,smote,96.097,-0.882,LOSS
1,f1,casb_relaxed,93.930,smote,95.366,-1.436,LOSS
2,precision,casb,98.750,smote,98.571,0.179,WIN
3,recall,casb_relaxed,91.429,smote,92.857,-1.429,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,98.164,smote,98.236,-0.073,LOSS
1,f1,casb,96.231,smote,96.459,-0.228,LOSS
2,precision,casb,95.938,smote,95.848,0.089,WIN
3,recall,casb_relaxed,97.500,smote,97.500,-0.000,TIE


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
1,42,1,svm,smote,0.75,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
2,42,1,svm,bsmote,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
3,42,1,svm,ros,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
4,42,1,svm,adasyn,0.75,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
5,42,1,svm,casb,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
6,42,1,svm,casb_relaxed,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
7,42,1,svm,casb_edited,0.75,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
8,42,1,svm,casb_ros,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
9,42,1,knn,original,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
